In [0]:
-- Validation checks for the initial Gold views.
-- Review every result before building cross-source marts.

-- Confirm that the expected Gold objects exist.
SHOW TABLES IN workforce_analytics.gold;


-- 1. dim_occupation must contain one row per SOC code.
SELECT
    soc_code,
    COUNT(*) AS row_count
FROM workforce_analytics.gold.dim_occupation
GROUP BY soc_code
HAVING COUNT(*) > 1;


-- 2. Required dimension fields should not be null.
SELECT
    SUM(
        CASE WHEN soc_code IS NULL THEN 1 ELSE 0 END
    ) AS null_soc_codes,
    SUM(
        CASE WHEN occupation_title IS NULL THEN 1 ELSE 0 END
    ) AS null_occupation_titles
FROM workforce_analytics.gold.dim_occupation;


-- 3. OEWS fact key should be unique.
SELECT
    series_id,
    year,
    period,
    COUNT(*) AS row_count
FROM workforce_analytics.gold.fact_oews_measurement
GROUP BY
    series_id,
    year,
    period
HAVING COUNT(*) > 1;


-- 4. Check how many OEWS records received readable lookup values.
SELECT
    COUNT(*) AS total_oews_rows,

    SUM(
        CASE WHEN occupation_name IS NOT NULL THEN 1 ELSE 0 END
    ) AS rows_with_occupation_name,

    SUM(
        CASE WHEN area_name IS NOT NULL THEN 1 ELSE 0 END
    ) AS rows_with_area_name,

    SUM(
        CASE WHEN industry_name IS NOT NULL THEN 1 ELSE 0 END
    ) AS rows_with_industry_name,

    SUM(
        CASE WHEN datatype_name IS NOT NULL THEN 1 ELSE 0 END
    ) AS rows_with_datatype_name

FROM workforce_analytics.gold.fact_oews_measurement;


-- 5. Projection records should match the occupation dimension.
SELECT
    COUNT(*) AS total_projection_rows,

    SUM(
        CASE WHEN occupation.soc_code IS NOT NULL THEN 1 ELSE 0 END
    ) AS matched_projection_rows,

    SUM(
        CASE WHEN occupation.soc_code IS NULL THEN 1 ELSE 0 END
    ) AS unmatched_projection_rows

FROM workforce_analytics.gold.fact_occupation_projection AS projection

LEFT JOIN workforce_analytics.gold.dim_occupation AS occupation
    ON projection.soc_code = occupation.soc_code;


-- 6. Education bridge business key should be unique.
SELECT
    onet_soc_code,
    element_id,
    scale_id,
    category,
    COUNT(*) AS row_count
FROM workforce_analytics.gold.bridge_occupation_education
GROUP BY
    onet_soc_code,
    element_id,
    scale_id,
    category
HAVING COUNT(*) > 1;


-- 7. Skill bridge business key should be unique.
SELECT
    onet_soc_code,
    skill_family,
    skill_id,
    scale_id,
    COUNT(*) AS row_count
FROM workforce_analytics.gold.bridge_occupation_skill
GROUP BY
    onet_soc_code,
    skill_family,
    skill_id,
    scale_id
HAVING COUNT(*) > 1;


-- 8. Software bridge business key should be unique.
SELECT
    onet_soc_code,
    software_name,
    element_id,
    COUNT(*) AS row_count
FROM workforce_analytics.gold.bridge_occupation_software
GROUP BY
    onet_soc_code,
    software_name,
    element_id
HAVING COUNT(*) > 1;


-- 9. Base-SOC coverage across O*NET Gold bridges.
-- Aggregate each bridge before joining to prevent many-to-many row expansion.
WITH education_coverage AS (
    SELECT
        soc_code,
        COUNT(DISTINCT onet_soc_code)
            AS onet_occupations_with_education
    FROM workforce_analytics.gold.bridge_occupation_education
    GROUP BY soc_code
),

skill_coverage AS (
    SELECT
        soc_code,
        COUNT(DISTINCT onet_soc_code)
            AS onet_occupations_with_skills
    FROM workforce_analytics.gold.bridge_occupation_skill
    GROUP BY soc_code
),

software_coverage AS (
    SELECT
        soc_code,
        COUNT(DISTINCT onet_soc_code)
            AS onet_occupations_with_software
    FROM workforce_analytics.gold.bridge_occupation_software
    GROUP BY soc_code
),

job_zone_coverage AS (
    SELECT
        soc_code,
        COUNT(DISTINCT onet_soc_code)
            AS onet_occupations_with_job_zone
    FROM workforce_analytics.gold.bridge_occupation_job_zone
    GROUP BY soc_code
)

SELECT
    occupation.soc_code,
    occupation.occupation_title,

    COALESCE(
        education.onet_occupations_with_education,
        0
    ) AS onet_occupations_with_education,

    COALESCE(
        skill.onet_occupations_with_skills,
        0
    ) AS onet_occupations_with_skills,

    COALESCE(
        software.onet_occupations_with_software,
        0
    ) AS onet_occupations_with_software,

    COALESCE(
        job_zone.onet_occupations_with_job_zone,
        0
    ) AS onet_occupations_with_job_zone

FROM workforce_analytics.gold.dim_occupation AS occupation

LEFT JOIN education_coverage AS education
    ON occupation.soc_code = education.soc_code

LEFT JOIN skill_coverage AS skill
    ON occupation.soc_code = skill.soc_code

LEFT JOIN software_coverage AS software
    ON occupation.soc_code = software.soc_code

LEFT JOIN job_zone_coverage AS job_zone
    ON occupation.soc_code = job_zone.soc_code

ORDER BY occupation.soc_code;
